# Week 2 — Day 4: Compare fine-tuning experiments

Tasks:
- Run 2-3 fine-tuning/config experiments (learning rates, decoding strategies) — done on Kaggle,
  see `notebooks/kaggle_week2_finetune_blip.ipynb`
- Log each run to MLflow; compare metrics across runs

Deliverable: MLflow dashboard with 3+ compared runs (zero-shot baseline + 3 fine-tuning configs);
a short note on which configuration performed best and why.

**Prerequisite:** run `notebooks/kaggle_week2_finetune_blip.ipynb` on Kaggle first, then download
`week2_finetune_results.json` (and `.csv`) from its Output tab into this repo's `data/processed/`
folder. This notebook reads that file — it will raise a clear error if it's not there yet.

### 1. Load the Kaggle fine-tuning results

In [1]:
import os
import json
import mlflow
import pandas as pd

results_path = '../results/week2_finetune_results.json'
if not os.path.exists(results_path):
    raise FileNotFoundError(
        "week2_finetune_results.json not found in results/.\n"
        "Run notebooks/kaggle_week2_finetune_blip.ipynb on Kaggle first, then download its "
        "output JSON/CSV into this repo's results/ folder before running this notebook."
    )

with open(results_path) as f:
    finetune_results = json.load(f)

finetune_df = pd.DataFrame(finetune_results).T
finetune_df

,lr,decoding,final_train_loss,train_time_s,bleu,rouge1,rouge2,rougeL
A_lr5e-5_greedy,0.00005,greedy,1.989793,299.958006,27.205776,54.04247,32.240746,50.471739
B_lr1e-5_greedy,0.00001,greedy,1.650241,304.883356,24.782337,52.802443,30.049554,49.563515
C_lr5e-5_beam,0.00005,beam,1.989793,299.958006,28.919218,55.324399,33.561844,52.246699


### 2. Log each fine-tuning config as an MLflow run

In [2]:
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("flickr8k-image-captioning")

logged_run_ids = {}
for config_name, row in finetune_df.iterrows():
    with mlflow.start_run(run_name=config_name) as run:
        mlflow.log_params({
            "model_name": "Salesforce/blip-image-captioning-base",
            "fine_tuned": True,
            "vision_encoder_frozen": True,
            "learning_rate": row["lr"],
            "decoding_strategy": row["decoding"],
            "trained_on": "kaggle-gpu",
        })
        mlflow.log_metrics({
            "bleu": row["bleu"],
            "rouge1": row["rouge1"],
            "rouge2": row["rouge2"],
            "rougeL": row["rougeL"],
            "final_train_loss": row["final_train_loss"],
            "train_time_s": row["train_time_s"],
        })
        logged_run_ids[config_name] = run.info.run_id

logged_run_ids

{'A_lr5e-5_greedy': 'cfe7ea78d83043d082f579a6935b0c6d',
 'B_lr1e-5_greedy': '93df6acbce5b4c88aab3cb0ed5c35f52',
 'C_lr5e-5_beam': '6c47aaea395b444a8a2ec767a0efabef'}

### 3. Compare all runs (zero-shot baseline + fine-tuned configs)

In [3]:
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name("flickr8k-image-captioning")
all_runs = client.search_runs([experiment.experiment_id], order_by=["metrics.bleu DESC"])

comparison = pd.DataFrame([
    {
        "run_name": r.data.tags.get("mlflow.runName", r.info.run_id[:8]),
        "fine_tuned": r.data.params.get("fine_tuned"),
        "decoding": r.data.params.get("decoding_strategy"),
        "lr": r.data.params.get("learning_rate", "-"),
        "bleu": r.data.metrics.get("bleu"),
        "rouge1": r.data.metrics.get("rouge1"),
        "rougeL": r.data.metrics.get("rougeL"),
    }
    for r in all_runs
])
comparison

,run_name,fine_tuned,decoding,lr,bleu,rouge1,rougeL
0,C_lr5e-5_beam,True,beam,5e-05,28.919218,55.324399,52.246699
1,A_lr5e-5_greedy,True,greedy,5e-05,27.205776,54.042470,50.471739
2,B_lr1e-5_greedy,True,greedy,1e-05,24.782337,52.802443,49.563515
3,zero-shot-blip-base,False,greedy,-,16.656270,50.194672,48.080152


### 4. Pick the baseline going forward

- **Best config: `C_lr5e-5_beam`** -- BLIP fine-tuned with lr=5e-5 (vision encoder frozen, text
  decoder only), evaluated with beam search (num_beams=4) instead of greedy decoding.
  BLEU 28.9, ROUGE-1 55.3, ROUGE-2 33.6, ROUGE-L 52.2.
- **Why:** it's the top scorer on all four metrics, not just one. It reuses the exact same
  fine-tuned weights as config A -- the only change is the decoding strategy at inference time --
  so the extra ~4-8% relative gain over A is essentially free (a few seconds more latency per
  caption, no extra training).
- **All three fine-tuned configs beat the zero-shot baseline by a wide margin** (BLEU 24.8-28.9 vs
  16.7, roughly +48% to +73% relative), confirming that even a light fine-tune (1 epoch, ~1,500
  images, vision encoder frozen) meaningfully adapts BLIP to Flickr8k's caption style.
- **Config B (lower LR) underperformed A**, despite reporting a lower `final_train_loss`. That
  loss value is the *last mini-batch's* loss, not an epoch average, so it's a noisy single-sample
  reading, not a reliable convergence signal -- the BLEU/ROUGE on the held-out eval images is the
  metric that matters, and by that measure A/C clearly beat B. With only 1,500 training images and
  1 epoch, the lower learning rate likely just adapted less to the domain in the available steps.
- **This becomes the Week 2 baseline model**: config A's fine-tuned checkpoint (`config_A_checkpoint`
  from the Kaggle run) used with beam-search decoding at inference. It's the model carried into
  Week 3's XAI analysis and registered in the MLflow Model Registry in Week 4.

To view all runs side-by-side in the UI:
```
.venv\\Scripts\\mlflow ui --backend-store-uri sqlite:///mlflow.db
```
then open http://127.0.0.1:5000